# 🔱 VoiceBatch Studio v2.2.7 - [Anti-Sleep & Auto-Split]
इसमें लंबी स्क्रिप्ट को छोटे टुकड़ों में प्रोसेस करके एक साथ जोड़ने का सिस्टम है।

In [ ]:
# @title 💤 Step 1: Anti-Sleep & Setup
import os
from IPython.display import display, Javascript

# एंटी-स्लीपिंग मोड चालू करना
display(Javascript('''
function ClickConnect(){ document.querySelector("colab-connect-button").click() }
setInterval(ClickConnect,60000)
'''))

print("⏳ जरूरी लाइब्रेरी लोड हो रही हैं...")
!pip install -q gradio librosa soundfile coqui-tts torchcodec
os.makedirs("outputs", exist_ok=True)
print("✅ इंजन तैयार है और कोलाब अब सोएगा नहीं!")

In [ ]:
# @title 🚀 Step 2: Launch app.py (Auto-Split Engine)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import re, os, numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def strict_hindi_filter(text):
    pattern = re.compile(r'[^\u0900-\u097F\s।,?!:;0-9]')
    return pattern.sub('', text)

def studio_pro_engine(text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    if lang == 'hi': text = strict_hindi_filter(text)
    
    # लंबी स्क्रिप्ट को वाक्यों में तोड़ना (Auto-Split)
    sentences = re.split(r'(?<=[।?!])\s+', text)
    combined_wav = []
    temp_output = 'outputs/temp_part.wav'
    final_path = 'outputs/VoiceBatch_Studio_Output.wav'
    
    print(f"🔄 कुल {len(sentences)} टुकड़ों में प्रोसेसिंग शुरू...")
    
    for part in sentences:
        if len(part.strip()) < 2: continue
        tts.tts_to_file(text=part, speaker_wav=audio_sample, language=lang, file_path=temp_output)
        y_part, sr = librosa.load(temp_output)
        combined_wav.extend(y_part)
    
    y = np.array(combined_wav)
    sr = 24000 # XTTS Default SR
    
    # पोस्ट प्रोसेसिंग
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(final_path, y, sr)
    return final_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.2.7')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Hindi Script (No Limit Now)', lines=10, placeholder='यहाँ अपनी लंबी कहानी लिखें...')
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr'], label='Language', value='hi')
            with gr.Row():
                spd = gr.Slider(0.7, 1.4, 1.0, label="Speed")
                ptc = gr.Slider(-4, 4, 0, label="Pitch")
            sil = gr.Checkbox(label="Silence Remover", value=True)
            btn = gr.Button('Generate Combined Long Audio ⚡', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Download: VoiceBatch_Studio_Output.wav')

    btn.click(studio_pro_engine, [txt, smp, spd, ptc, lng, sil], out)
demo.launch(share=True, debug=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप तैयार है! अब लॉन्च हो रहा है...")
!python app.py

In [ ]:
# @title 📁 Step 3: तत्काल डाउनलोड
from google.colab import files
import os
file_path = 'outputs/VoiceBatch_Studio_Output.wav'
if os.path.exists(file_path):
    files.download(file_path)
    print("✅ डाउनलोड शुरू हो गया है।")
else:
    print("⚠️ ऑडियो फाइल नहीं मिली। पहले Step 2 में ऑडियो जनरेट करें।")